# Baseline training (CLIP-B/16 and SigLIP-2)

This notebook trains two frozen-backbone baselines using precomputed visual features
and BGE-M3 query encoding, with symmetric InfoNCE loss.

In [1]:
import gc
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

cwd = Path.cwd().resolve()
if (cwd / 'source').exists():
    BASE_DIR = cwd
elif (cwd.parent / 'source').exists():
    BASE_DIR = cwd.parent
else:
    raise RuntimeError(f'Cannot locate project root from cwd={cwd}')

SOURCE_DIR = BASE_DIR / 'source'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('BASE_DIR:', BASE_DIR)
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR: /media/urlab/KINGSTON/aic
DEVICE: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


In [2]:
from dataset import get_dataloader

OUTPUT_DIR = SOURCE_DIR / 'baseline_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BGEM3_FEATURE_DIR = BASE_DIR / 'features' / 'bgem3'
BGEM3_PATH = BASE_DIR / 'features' / 'weights' / 'bgem3'

if not BGEM3_FEATURE_DIR.exists():
    raise FileNotFoundError(f'BGE-M3 features not found: {BGEM3_FEATURE_DIR}')
if not BGEM3_PATH.exists():
    raise FileNotFoundError(f'BGE-M3 weights not found: {BGEM3_PATH}')

CLIP_FEATURE_SUBDIR = 'clip_vitb16_l2'
SIGLIP_FEATURE_SUBDIR = 'siglip2'

CFG = {
    'max_frames': 25,
    'max_segments': 15,
    'max_seg_tokens': 128,
    'batch_size': 8,
    'num_workers': os.cpu_count() or 4,
    'epochs': 50,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'grad_clip': 1.0,
    'train_query_max_length': 512,
    'eval_query_max_length': 256,
    'eval_query_batch_size': 16,
    'dual_softmax_tau': 0.01,
}

print('OUTPUT_DIR:', OUTPUT_DIR)
print('BGEM3_FEATURE_DIR:', BGEM3_FEATURE_DIR)
print('BGEM3_PATH:', BGEM3_PATH)
print('CLIP_FEATURE_SUBDIR:', CLIP_FEATURE_SUBDIR)
print('SIGLIP_FEATURE_SUBDIR:', SIGLIP_FEATURE_SUBDIR)


OUTPUT_DIR: /media/urlab/KINGSTON/aic/source/baseline_output
BGEM3_FEATURE_DIR: /media/urlab/KINGSTON/aic/features/bgem3
BGEM3_PATH: /media/urlab/KINGSTON/aic/features/weights/bgem3
CLIP_FEATURE_SUBDIR: clip_vitb16_l2
SIGLIP_FEATURE_SUBDIR: siglip2


In [3]:
def _feature_dir_exists(subdir):
    return (BASE_DIR / 'features' / subdir).exists()

def build_loaders(visual_feature_subdir):
    return (
        get_dataloader(
            split='train',
            base_dir=BASE_DIR,
            batch_size=CFG['batch_size'],
            num_workers=CFG['num_workers'],
            max_frames=CFG['max_frames'],
            max_segments=CFG['max_segments'],
            max_seg_tokens=CFG['max_seg_tokens'],
            visual_feature_subdir=visual_feature_subdir,
        ),
        get_dataloader(
            split='val',
            base_dir=BASE_DIR,
            batch_size=CFG['batch_size'],
            num_workers=CFG['num_workers'],
            max_frames=CFG['max_frames'],
            max_segments=CFG['max_segments'],
            max_seg_tokens=CFG['max_seg_tokens'],
            visual_feature_subdir=visual_feature_subdir,
        ),
    )

clip_train_loader = clip_val_loader = None
if _feature_dir_exists(CLIP_FEATURE_SUBDIR):
    clip_train_loader, clip_val_loader = build_loaders(CLIP_FEATURE_SUBDIR)
    print('CLIP train:', len(clip_train_loader.dataset), 'val:', len(clip_val_loader.dataset))
else:
    print('CLIP features not found:', BASE_DIR / 'features' / CLIP_FEATURE_SUBDIR)

siglip_train_loader = siglip_val_loader = None
if _feature_dir_exists(SIGLIP_FEATURE_SUBDIR):
    siglip_train_loader, siglip_val_loader = build_loaders(SIGLIP_FEATURE_SUBDIR)
    print('SigLIP train:', len(siglip_train_loader.dataset), 'val:', len(siglip_val_loader.dataset))
else:
    print('SigLIP features not found:', BASE_DIR / 'features' / SIGLIP_FEATURE_SUBDIR)

CLIP train: 9222 val: 9222
SigLIP train: 9222 val: 9222


In [4]:
bgem3_tokenizer = AutoTokenizer.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model = AutoModel.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model.eval().to(DEVICE)
for p in bgem3_model.parameters():
    p.requires_grad = False
print('BGE-M3 loaded from:', BGEM3_PATH)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 843.15it/s, Materializing param=pooler.dense.weight]                               


BGE-M3 loaded from: /media/urlab/KINGSTON/aic/features/weights/bgem3


In [5]:
def encode_queries(texts, tokenizer, encoder_model, device, max_length=512, batch_size=None):
    if not texts:
        return torch.empty((0, encoder_model.config.hidden_size), device=device)
    if batch_size is None:
        batch_size = len(texts)
    all_embs = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            chunk = texts[start:start + batch_size]
            enc = tokenizer(
                chunk, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt',
            ).to(device)
            if device.type == 'cuda':
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    h = encoder_model(**enc, return_dict=True).last_hidden_state
            else:
                h = encoder_model(**enc, return_dict=True).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp_min(1e-6)
            all_embs.append(F.normalize(pooled.float(), p=2, dim=-1))
    return torch.cat(all_embs, dim=0)

In [6]:
def _masked_mean(x, mask, dim):
    m = mask.unsqueeze(-1).to(x.dtype)
    return (x * m).sum(dim=dim) / m.sum(dim=dim).clamp_min(1e-6)

class BaselineVideoTextModel(nn.Module):
    def __init__(self, vis_dim, out_dim=1024, init_temp=0.07):
        super().__init__()
        self.proj = nn.Linear(vis_dim, out_dim, bias=False)
        self.temperature = nn.Parameter(torch.log(torch.tensor(1.0 / init_temp)))

    def forward(self, visual_features, visual_mask):
        vis = visual_features.mean(dim=2)
        vis_mean = _masked_mean(vis, visual_mask, dim=1)
        v = self.proj(vis_mean)
        v = F.normalize(v, p=2, dim=-1)
        return v, self.temperature

def precompute_val_docs(model, loader, device):
    model.eval()
    doc_embs, shot_ids = [], []
    with torch.inference_mode():
        for batch in tqdm(loader, desc='Val docs', leave=False, dynamic_ncols=True):
            vis = batch['visual_features'].to(device)
            vmask = batch['visual_mask'].to(device)
            with torch.amp.autocast(
                device_type='cuda' if device.type == 'cuda' else 'cpu',
                enabled=(device.type == 'cuda'), dtype=torch.bfloat16,
            ):
                v, _ = model(vis, vmask)
            doc_embs.append(v.detach().cpu())
            shot_ids.extend(batch['shot_id'])
    return F.normalize(torch.cat(doc_embs, dim=0), p=2, dim=-1), shot_ids

def load_val_queries():
    q_items = []
    for jf in sorted((BASE_DIR / 'data' / 'val').glob('*.json')):
        video_id = jf.stem
        rows = json.loads(jf.read_text('utf-8'))
        for row in rows:
            sid = str(row.get('id', '')).zfill(3)
            q = str(row.get('positive', '')).strip()
            if sid and q:
                q_items.append({'shot_id': f'{video_id}_{sid}', 'query': q})
    seen, deduped = set(), []
    for x in q_items:
        if x['shot_id'] not in seen:
            seen.add(x['shot_id'])
            deduped.append(x)
    return deduped

def evaluate_on_val(model, loader, device, tau=None):
    doc_embs, shot_ids = precompute_val_docs(model, loader, device)
    q_items = load_val_queries()
    q_embs_all = encode_queries(
        [x['query'] for x in q_items],
        bgem3_tokenizer, bgem3_model, device,
        max_length=CFG['eval_query_max_length'],
        batch_size=CFG['eval_query_batch_size'],
    )
    shot_to_idx = {s: i for i, s in enumerate(shot_ids)}
    filtered = [x for x in q_items if x['shot_id'] in shot_to_idx]
    q_idx = [q_items.index(x) for x in filtered]
    q_embs_f = q_embs_all[q_idx]
    gt_indices = [shot_to_idx[x['shot_id']] for x in filtered]

    if tau is None:
        tau = torch.exp(model.temperature).item()

    sim_raw = torch.matmul(q_embs_f, doc_embs.to(device).T) / tau
    sim_dsl = torch.softmax(sim_raw, dim=1) * torch.softmax(sim_raw, dim=0)
    sim_np = sim_dsl.detach().cpu().numpy()
    ranks = [
        int(np.where(np.argsort(-sim_np[i]) == gt)[0][0]) + 1
        for i, gt in enumerate(gt_indices)
    ]
    r = np.asarray(ranks)
    return {
        'R1': 100.0 * float(np.mean(r <= 1)),
        'R5': 100.0 * float(np.mean(r <= 5)),
        'R10': 100.0 * float(np.mean(r <= 10)),
        'MdR': float(np.median(r)),
        'SumR': 100.0 * float(np.mean(r <= 1) + np.mean(r <= 5) + np.mean(r <= 10)),
    }


In [7]:
def symmetric_infonce(q, v, temperature_param):
    tau = torch.exp(temperature_param).clamp_min(1e-6)
    sim = torch.matmul(q, v.T) / tau
    labels = torch.arange(sim.size(0), device=sim.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels))

def infer_vis_dim(loader, expected=None, name=''):
    batch = next(iter(loader))
    actual = int(batch['visual_features'].shape[-1])
    if expected is not None and actual != expected:
        print(f'[WARN] {name} vis_dim={actual} differs from expected {expected}. Using {actual}.')
    return actual


In [8]:
def train_baseline(model, train_loader, val_loader, desc, ckpt_name):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay']
    )
    epochs = CFG['epochs']
    total_steps = max(1, len(train_loader) * epochs)
    warmup_steps = int(total_steps * CFG['warmup_ratio'])
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.amp.GradScaler('cuda' if DEVICE.type == 'cuda' else 'cpu')

    ckpt_path = OUTPUT_DIR / ckpt_name
    log_path = OUTPUT_DIR / ckpt_name.replace('.pth', '_log.json')

    best_r1 = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        pbar = tqdm(
            train_loader, total=len(train_loader),
            desc=f'[{desc}] Epoch {epoch:02d}/{epochs}',
            leave=False, dynamic_ncols=True,
        )
        for step, batch in enumerate(pbar, start=1):
            q_hat = encode_queries(
                batch['query_text'], bgem3_tokenizer, bgem3_model, DEVICE,
                max_length=CFG['train_query_max_length'],
            )
            vis = batch['visual_features'].to(DEVICE)
            vmask = batch['visual_mask'].to(DEVICE)

            with torch.amp.autocast(
                device_type='cuda' if DEVICE.type == 'cuda' else 'cpu',
                enabled=(DEVICE.type == 'cuda'), dtype=torch.bfloat16,
            ):
                v_hat, temperature = model(vis, vmask)
                loss = symmetric_infonce(q_hat, v_hat, temperature)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += float(loss.item())
            pbar.set_postfix(loss=f'{running_loss / step:.4f}')

        train_loss = running_loss / len(train_loader)

        if DEVICE.type == 'cuda':
            gc.collect()
            torch.cuda.empty_cache()

        val_metrics = evaluate_on_val(model, val_loader, DEVICE, tau=None)
        history.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics})

        r1 = val_metrics['R1']
        r5 = val_metrics['R5']
        r10 = val_metrics['R10']
        if r1 > best_r1:
            best_r1 = r1
            torch.save(model.state_dict(), ckpt_path)

        print(
            f'[{desc}] Epoch {epoch:02d}/{epochs} '
            f'| loss={train_loss:.4f} '
            f'| val R@1={r1:.2f} '
            f'| R@5={r5:.2f} '
            f'| R@10={r10:.2f}'
        )

        log_path.write_text(json.dumps(history, indent=2), encoding='utf-8')

    print(f'[{desc}] Best Val R@1: {best_r1:.2f} | Saved: {ckpt_path}')
    return history, best_r1

In [9]:
if clip_train_loader is None or clip_val_loader is None:
    raise RuntimeError('CLIP features not ready. Run precompute_clip_patch16.py first.')

clip_vis_dim = infer_vis_dim(clip_train_loader, expected=512, name='CLIP')
clip_model = BaselineVideoTextModel(vis_dim=clip_vis_dim, out_dim=1024)
history_clip, best_r1_clip = train_baseline(
    clip_model, clip_train_loader, clip_val_loader,
    desc='CLIP-B16',
    ckpt_name='baseline_clip_b16.pth',
)

[WARN] CLIP vis_dim=768 differs from expected 512. Using 768.


[CLIP-B16] Epoch 01/50 | loss=2.0778 | val R@1=0.33 | R@5=1.76 | R@10=2.95


[CLIP-B16] Epoch 02/50 | loss=2.0732 | val R@1=2.15 | R@5=7.67 | R@10=11.68


[CLIP-B16] Epoch 03/50 | loss=2.0685 | val R@1=4.08 | R@5=12.32 | R@10=18.51


[CLIP-B16] Epoch 04/50 | loss=2.0658 | val R@1=5.18 | R@5=15.15 | R@10=22.05


...

[CLIP-B16] Epoch 46/50 | loss=1.7287 | val R@1=18.22 | R@5=38.36 | R@10=48.99


[CLIP-B16] Epoch 47/50 | loss=1.7292 | val R@1=18.22 | R@5=38.44 | R@10=48.99


[CLIP-B16] Epoch 48/50 | loss=1.7297 | val R@1=18.23 | R@5=38.42 | R@10=49.06


[CLIP-B16] Epoch 49/50 | loss=1.7293 | val R@1=18.22 | R@5=38.42 | R@10=49.02


[CLIP-B16] Epoch 50/50 | loss=1.7275 | val R@1=18.23 | R@5=38.42 | R@10=49.01
[CLIP-B16] Best Val R@1: 18.23 | Saved: /media/urlab/KINGSTON/aic/source/baseline_output/baseline_clip_b16.pth


In [ ]:
if siglip_train_loader is None or siglip_val_loader is None:
    raise RuntimeError('SigLIP features not ready. Run precompute_siglip2.py first.')

siglip_vis_dim = infer_vis_dim(siglip_train_loader, expected=1152, name='SigLIP2')
siglip_model = BaselineVideoTextModel(vis_dim=siglip_vis_dim, out_dim=1024)
history_siglip, best_r1_siglip = train_baseline(
    siglip_model, siglip_train_loader, siglip_val_loader,
    desc='SigLIP2',
    ckpt_name='baseline_siglip2.pth',
)

[SigLIP2] Epoch 01/50 | loss=2.0772 | val R@1=1.41 | R@5=5.43 | R@10=8.87


[SigLIP2] Epoch 02/50 | loss=2.0703 | val R@1=4.54 | R@5=14.12 | R@10=21.00


[SigLIP2] Epoch 03/50 | loss=2.0664 | val R@1=7.64 | R@5=20.65 | R@10=28.99


[SigLIP2] Epoch 04/50 | loss=2.0642 | val R@1=10.23 | R@5=25.83 | R@10=34.71


[SigLIP2] Epoch 05/50 | loss=2.0619 | val R@1=11.60 | R@5=27.77 | R@10=37.39


...

[SigLIP2] Epoch 47/50 | loss=1.7039 | val R@1=25.61 | R@5=47.61 | R@10=57.67


[SigLIP2] Epoch 48/50 | loss=1.7019 | val R@1=25.66 | R@5=47.52 | R@10=57.63


[SigLIP2] Epoch 49/50 | loss=1.7018 | val R@1=25.66 | R@5=47.56 | R@10=57.62


[SigLIP2] Epoch 50/50 | loss=1.7027 | val R@1=25.67 | R@5=47.54 | R@10=57.64
[SigLIP2] Best Val R@1: 25.67 | Saved: /media/urlab/KINGSTON/aic/source/baseline_output/baseline_siglip2.pth


: 